In [1]:
import os
from pathlib import Path

root = Path("/kaggle/input")
for d in root.iterdir():
    if d.is_dir():
        print("DATASET:", d)
        # show top-level
        print("  top:", [x.name for x in d.iterdir() if x.is_dir()][:10])
        # if raw-img exists, show its class folders
        raw = d / "raw-img"
        if raw.exists():
            print("  raw-img classes:", [x.name for x in raw.iterdir() if x.is_dir()][:20])
        print()

DATASET: /kaggle/input/datasets
  top: ['viratkothari']



In [2]:
# from pathlib import Path

# root = Path("/kaggle/input")

# for p in root.rglob("*"):
#     if p.is_dir():
#         print(p)

In [3]:
# from pathlib import Path

# dataset = Path("/kaggle/input/datasets/viratkothari/animal10/raw-img")

# print("Classes:")
# print([x.name for x in dataset.iterdir()])


In [4]:
# =========================
# Phase 1 (Level 1) - Animal10 Training (Kaggle GPU)
# Dataset: /kaggle/input/datasets/viratkothari/animal10/Animals-10
# =========================

import os, json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix
import timm
from tqdm import tqdm

In [5]:
# ---------- Repro ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------- Paths ----------
DATA_ROOT = "/kaggle/input/datasets/viratkothari/animal10/Animals-10"
OUT_DIR = Path("/kaggle/working/level1_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

Device: cuda


In [6]:

# ---------- Hyperparams (kids mode) ----------
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 5
LR = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "efficientnet_b0"

# ---------- Transforms ----------
train_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])

val_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
])

In [7]:

# ---------- Build datasets (auto-split 80/20) ----------
full_train = ImageFolder(DATA_ROOT, transform=train_tf)
full_val   = ImageFolder(DATA_ROOT, transform=val_tf)

n = len(full_train)
idx = np.arange(n)
np.random.shuffle(idx)
val_ratio = 0.2
val_n = int(n * val_ratio)

val_idx = idx[:val_n]
train_idx = idx[val_n:]

train_ds = Subset(full_train, train_idx)
val_ds   = Subset(full_val, val_idx)

class_to_idx = full_train.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
num_classes = len(class_to_idx)

print("Total images:", n)
print("Train:", len(train_ds), "Val:", len(val_ds))
print("Classes:", num_classes, list(class_to_idx.keys()))


Total images: 26179
Train: 20944 Val: 5235
Classes: 10 ['butterfly', 'cat', 'chicken', 'cow', 'dog', 'elephant', 'horse', 'sheep', 'spider', 'squirrel']


In [8]:

# ---------- Loaders ----------
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


In [9]:

# ---------- Model ----------
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [10]:

# ---------- Metrics ----------
@torch.no_grad()
def eval_metrics():
    model.eval()
    y_true, y_pred = [], []
    top3_correct = 0
    total = 0

    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)

        # top-1
        pred1 = torch.argmax(logits, dim=1)

        # top-3
        top3 = torch.topk(logits, k=3, dim=1).indices
        top3_correct += (top3 == y.unsqueeze(1)).any(dim=1).sum().item()

        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred1.cpu().tolist())
        total += y.size(0)

    top1 = (np.array(y_true) == np.array(y_pred)).mean()
    top3_acc = top3_correct / max(total, 1)
    cm = confusion_matrix(y_true, y_pred)
    return float(top1), float(top3_acc), cm

In [11]:
# ---------- Train ----------
best_top1 = -1.0
best_path = OUT_DIR / "level1_model_best.pt"

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0
    t0 = time.time()

    for x, y in tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}"):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = x.size(0)
        running_loss += loss.item() * bs
        seen += bs

    train_loss = running_loss / max(seen, 1)
    top1, top3, cm = eval_metrics()
    dt = time.time() - t0

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} top1={top1:.4f} top3={top3:.4f} time={dt:.1f}s")

    if top1 > best_top1:
        best_top1 = top1
        torch.save(model.state_dict(), best_path)
        print("✅ Saved best:", best_path)


epoch 1/5: 100%|██████████| 328/328 [02:40<00:00,  2.04it/s]


Epoch 1: train_loss=0.2639 top1=0.9593 top3=0.9922 time=183.8s
✅ Saved best: /kaggle/working/level1_outputs/level1_model_best.pt


epoch 2/5: 100%|██████████| 328/328 [01:50<00:00,  2.96it/s]


Epoch 2: train_loss=0.0729 top1=0.9643 top3=0.9956 time=122.4s
✅ Saved best: /kaggle/working/level1_outputs/level1_model_best.pt


epoch 3/5: 100%|██████████| 328/328 [01:50<00:00,  2.96it/s]


Epoch 3: train_loss=0.0430 top1=0.9687 top3=0.9947 time=122.4s
✅ Saved best: /kaggle/working/level1_outputs/level1_model_best.pt


epoch 4/5: 100%|██████████| 328/328 [01:50<00:00,  2.97it/s]


Epoch 4: train_loss=0.0449 top1=0.9618 top3=0.9939 time=122.1s


epoch 5/5: 100%|██████████| 328/328 [01:51<00:00,  2.94it/s]


Epoch 5: train_loss=0.0387 top1=0.9637 top3=0.9922 time=123.4s


In [12]:




# ---------- Export artifacts ----------
last_path = OUT_DIR / "level1_model_last.pt"
torch.save(model.state_dict(), last_path)

(OUT_DIR / "class_to_idx.json").write_text(json.dumps(class_to_idx, indent=2))

metrics = {
    "best_val_top1": best_top1,
    "model_name": MODEL_NAME,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
}
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))

config_snapshot = {
    "seed": SEED,
    "data_root": DATA_ROOT,
    "val_ratio": val_ratio,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "model_name": MODEL_NAME,
    "num_classes": num_classes,
}
(OUT_DIR / "config_level1.json").write_text(json.dumps(config_snapshot, indent=2))

print("\n✅ DONE. Exported files:")
for p in OUT_DIR.iterdir():
    print(" -", p)


✅ DONE. Exported files:
 - /kaggle/working/level1_outputs/level1_model_last.pt
 - /kaggle/working/level1_outputs/metrics.json
 - /kaggle/working/level1_outputs/config_level1.json
 - /kaggle/working/level1_outputs/level1_model_best.pt
 - /kaggle/working/level1_outputs/class_to_idx.json
